In [ ]:
from dotenv import load_dotenv
import os

load_dotenv()
os.environ["HF_TOKEN"] = os.getenv("HF_TOKEN")
os.environ["TAVILY_API_KEY"] = os.getenv("TAVILY_API_KEY")
os.environ["GROQ_API_KEY"] = os.getenv("GROQ_API_KEY")

#export TAVILY_API_KEY="your-api-key"

get llm

In [ ]:
# from langchain_community.tools import wikipedia, TavilySearchResults  - its deprecated
import langchain
# from langchain_huggingface import HuggingFaceEndpoint,ChatHuggingFace
from langchain_groq import ChatGroq


#ChatHuggingFace — a LangChain adapter that wraps that HF endpoint so it behaves like a chat-style model (adds proper chat message formatting, system/user/assistant roles, etc.) on top of the raw HuggingFaceEndpoint.
# llm = HuggingFaceEndpoint(
#     repo_id="Qwen/Qwen2.5-7B-Instruct",
#      provider="hf-inference",  # force HF's own inference instead of a 3rd party
#     temperature=0
# )
# chat_model = ChatHuggingFace(llm=llm)

# llm = HuggingFaceEndpoint(
#     repo_id="openai/gpt-oss-20b",   # or "openai/gpt-oss-120b" for the larger one
#     temperature=0
# )
chat_model = ChatGroq(
    # model="llama-3.3-70b-versatile",#gave access issue 
    model = "openai/gpt-oss-20b",
    temperature=0
)


langchain.debug = True




1st tool - web result using travily search

alternate for travily :
DuckDuckGo — free, no API key at all   => from duckduckgo_search import DDGS <br>
Serper.dev — Google results, has an official LangChain wrapper => pip install google-search-results  # or use langchain's GoogleSerperAPIWrapper
Brave Search API
Exa — "search engine built for AI"

Note : DuckDuckGo as your primary, since it has literally no quota to run out of and no signup friction, and keep Tavily as a secondary option


In [ ]:
from langchain_tavily import TavilySearch

web_search = TavilySearch(max_results = 1, include_answer =True, include_raw_content = True, include_images = True)
answer = web_search.invoke({"query":"Bangalore weather today?"})
print(type(answer),answer)
print(answer.keys())



2nd tool - wikipedia

In [ ]:
from langchain_core.tools import tool
import wikipedia
import wikipedia.wikipedia as wiki_internal
import requests

wiki_internal.USER_AGENT = "MyLangchainApp/1.0 (myemail@example.com)"

@tool
def wikipediaSearch(query:str)->str:
    """Search Wikipedia for information about a topic, person, or historical event."""
    try:
        return wikipedia.summary(query,sentences = 3)
    except wikipedia.exceptions.DisambiguationError as e:
        return f"Ambigous questions"
    except wikipedia.exceptions.PageError :
        return "No wikipedia page found for matching this query"
    except (requests.exceptions.JSONDecodeError, requests.exceptions.RequestException) as e:
        return f"Wikipedia request failed, try again: {e}"
    

wikiAnswer = wikipediaSearch.invoke({"query":"India independence day"})
print(wikiAnswer)



Create tool list and agent

Name	Package	Status <br>
create_react_agent + AgentExecutor	langchain_classic.agents	Legacy, text-parsing based<br>
create_react_agent	langgraph.prebuilt	Current, tool-calling based<br>
create_agent	langchain.agents	Current top-level convenience wrapper (LangGraph under the hood)<br>

In [ ]:
from langchain.agents import create_agent
import time

# tool list 

tools = [wikipediaSearch,web_search]

query = "How is weather in Bangalore today? Please tell when India got independense."

agent = create_agent(model=chat_model,tools = tools)

for attempt in range(3):
    try:
        langchain.debug = True
        result = agent.invoke({"messages":[{"role":"user","content":query}]})
        langchain.debug = False
    except Exception as e:
        print(f"Attempt {attempt+1} is failed :{e}")
        time.sleep(5)
# result = agent.invoke({"query":query})
print(result["messages"][-1].content)

In [ ]:
type(result["messages"][-1])
result["messages"][-1]


Tools are available options for the model, not steps it's required to execute. If the model is confident it already knows something (like "India's independence day" — a very well-known fact, likely deeply embedded in its training data), it can reasonably skip calling the Wikipedia tool and just answer directly from its own knowledge. So, it did not give wikipedia as tool.

In [ ]:
for msg in result["messages"]:
    if hasattr(msg, "tool_calls") and msg.tool_calls:
        print("Called:", [tc["name"] for tc in msg.tool_calls])
    if msg.type == "tool":
        print(f"  → {msg.name} returned: {msg.content}")

In [ ]:
print(type(chat_model))
print(chat_model)

In [ ]:
langchain.debug = True
result = agent.invoke({"messages":[{"role":"user","content":"How is weather today in Bangalore?First Chief Minister of Karnataka"}]})
langchain.debug = False


when wikipedia returns 1 sentence :

Called: ['wikipediaSearch']
  → wikipediaSearch returned: The Chief Minister of Karnataka is the head of government of the Indian state of Karnataka.
Called: ['wikipediaSearch']
  → wikipediaSearch returned: The Chief Minister of Karnataka is the head of government of the Indian state of Karnataka.
Called: ['wikipediaSearch']
  → wikipediaSearch returned: No wikipedia page found for matching this query
Called: ['wikipediaSearch']
  → wikipediaSearch returned: Kyasamballi Chengaluraya Reddy (4 May 1902 – 27 February 1976) was the first Chief Minister of Mysore State (Now Karnataka).

  first call to wiki did not get proper answer, llm tweaks query and search, not again, at the end it gets answer.
  

In [ ]:
for msg in result["messages"]:
    if hasattr(msg,"tool_calls") and msg.tool_calls:
        print("Called:", [tc["name"] for tc in msg.tool_calls])
    if msg.type == "tool":
        print(f"  → {msg.name} returned: {msg.content}")
         